In [1]:
import hashlib
import random 
import time 
from fastapi import FastAPI
from prometheus_client import Counter , Histogram , make_asgi_app

In [2]:
app = FastAPI()

In [3]:
# track metrics separately per model version 
REQUEST_COUNT = Counter(
    'ab_request_total',
    'Total requests per model version',
    ['model_version', 'status']
)

PREDICTION_LATENCY = Histogram(
    'ab_prediction_latency_seconds',
    'Latency per model version',
    ['model_version'],
    buckets=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
)

FRAUD_RATE = Counter(
    'ab_fraud_predictions_total',
    'Fraud predictions per model version',
    ['model_version']
)

In [4]:
metrics_app = make_asgi_app()
app.mount("/metrics",metrics_app)

In [5]:
def get_model_version(user_id: str, traffic_split: int = 10) -> str:
    has_value = int(hashlib.md5(user_id.encode()).hexdigest(),16)
    if hash_value % 100 <traffic_split:
        return "model_b"   # <- 10 % users
    return "model_a"   #  <- 90% users

In [6]:
def model_a_predict(features: dict) -> dict:
    time.sleep(random.uniform(0.001,0.005))
    is_fraud = random.random() < 0.03
    return {"fraud":is_fraud, "model": "model_a", "confidence": 0.85}

In [7]:
def model_b_predict(features: dict) -> dict:
    time.sleep(random.uniform(0.001,0.008))
    is_fraud = random.random() < 0.03
    return {"fraud":is_fraud, "model": "model_a", "confidence": 0.91}

In [8]:
@app.get("/predict/{user_id}")
def predict(user_id: str):
    start_time = time.time()

    version = get_model_version(user_id)

    try:
        if version == "model_b":
            result = model_b_predict({})
        else:
            result = model_a_predict({})

        latency = time.time() - start_time

        PREDICTION_LATENCY.labels(model_version=version).observe(latency)
        REQUEST_COUNT.labels(model_version=version, status='success').inc()
        if result['fraud']:
            FRAUD_RATE.labels(model_version=version).inc()
        return result

    except Exception as e:
        REQUEST_COUNT.labels(model_version=version, status= 'error' ).inc()
        raise e

## Summary

**What I built:** A/B testing framework — hash-based traffic routing
between two model versions with per-version Prometheus metrics.

**Key concepts:**
- A/B test: split traffic between old model (control) and new model (treatment)
- Hash-based routing: hash(user_id) % 100 < split_percentage
- Deterministic: same user always gets same model — debuggable
- Random routing: same user gets different models — not debuggable

**Why not swap all traffic at once:**
- Test metrics are optimistic — clean sample, not real production diversity
- New model might be worse on specific user segments
- Full swap = all users affected if model_b fails
- A/B test = only 10% affected, instant rollback

**Traffic split progression:**
- Start: 10% model_b, 90% model_a
- model_b wins → 25% → 50% → 100%
- model_b loses on any metric → rollback instantly

**Metrics to compare per version:**
- fraud_detection_rate
- false_positive_rate  
- p99_latency
- error_rate

**Key insight:** Hash user_id not random — deterministic routing
means same user always gets same model. Debugging, logging, and
rollback all become tractable. Random routing makes all three impossible.

**Next:** Week 7 review — Day 49